## Example 1 — compile to machine code with MCJIT and call it

In [ ]:
import ctypes
from llvmlite import binding as llvm

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()
module = llvm.parse_assembly("""
define i64 @sum_to(i64 %n) {
entry:
  br label %loop
loop:
  %i = phi i64 [0, %entry], [%i.next, %loop]
  %acc = phi i64 [0, %entry], [%acc.next, %loop]
  %acc.next = add i64 %acc, %i
  %i.next = add i64 %i, 1
  %done = icmp ugt i64 %i.next, %n
  br i1 %done, label %exit, label %loop
exit:
  ret i64 %acc.next
}
""")
machine = llvm.Target.from_default_triple().create_target_machine()
engine = llvm.create_mcjit_compiler(module, machine)
engine.finalize_object()
sum_to = ctypes.CFUNCTYPE(ctypes.c_int64, ctypes.c_int64)(engine.get_function_address("sum_to"))
assert sum_to(100) == 5050, sum_to(100)
print("OK -- MCJIT-compiled sum_to(100) =", sum_to(100), "on", llvm.get_host_cpu_name())

## Example 2 — the optimiser (new pass manager) folds a constant expression

In [ ]:
from llvmlite import binding as llvm

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()
module = llvm.parse_assembly("""
define i32 @answer() {
  %slot = alloca i32
  store i32 6, ptr %slot
  %six = load i32, ptr %slot
  %product = mul i32 %six, 7
  ret i32 %product
}
""")
machine = llvm.Target.from_default_triple().create_target_machine()
builder = llvm.create_pass_builder(machine, llvm.create_pipeline_tuning_options(speed_level=2))
builder.getModulePassManager().run(module, builder)
body = str(module.get_function("answer"))
assert "ret i32 42" in body and "alloca" not in body and "mul" not in body, body
print("OK -- O2 folded the function to:", [l.strip() for l in body.splitlines() if l.strip().startswith("ret")][0])

## Example 3 — round trips: bitcode, and an object file read back section by section

In [ ]:
from llvmlite import binding as llvm

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()
module = llvm.parse_assembly("define i32 @seven() {\n  ret i32 7\n}\n")
bitcode = module.as_bitcode()
assert bitcode[:2] == b"BC", bitcode[:4]
again = llvm.parse_bitcode(bitcode)
assert str(again.get_function("seven")) == str(module.get_function("seven"))
machine = llvm.Target.from_default_triple().create_target_machine()
assert "seven:" in machine.emit_assembly(module)
obj = machine.emit_object(module)
assert obj[:4] == b"\x7fELF", obj[:4]
sections = {s.name(): s.size() for s in llvm.ObjectFileRef.from_data(obj).sections()}
assert sections.get(b".text", 0) > 0, sections
print(f"OK -- bitcode {len(bitcode)} bytes round-trips; object {len(obj)} bytes, .text {sections[b'.text']} bytes")

## Example 4 — value kinds, and the ORC JIT

In [ ]:
import ctypes
from llvmlite import binding as llvm

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()
module = llvm.parse_assembly("define double @half(double %x) {\n  %r = fmul double %x, 5.000000e-01\n  ret double %r\n}\n")
function = module.get_function("half")
instruction = next(iter(next(iter(function.blocks)).instructions))
kinds = [operand.value_kind for operand in instruction.operands]
assert kinds == [llvm.ValueKind.argument, llvm.ValueKind.constant_fp], kinds
assert function.value_kind == llvm.ValueKind.function and instruction.value_kind == llvm.ValueKind.instruction
# The ORC JIT (LLJIT).
machine = llvm.Target.from_default_triple().create_target_machine(jit=True, reloc="default")
lljit = llvm.create_lljit_compiler(machine)
tracker = llvm.JITLibraryBuilder().add_ir(str(module)).export_symbol("half").add_current_process().link(lljit, "half_lib")
half = ctypes.CFUNCTYPE(ctypes.c_double, ctypes.c_double)(tracker["half"])
assert half(21.0) == 10.5, half(21.0)
print("OK -- value kinds read", [k.name for k in kinds], "and LLJIT-compiled half(21.0) =", half(21.0))

## Example 5 — one printed instruction names the same debug metadata as the printed module

In [ ]:
import re
from llvmlite import binding as llvm

# The `!dbg !N` in str(instruction) must be the node that str(module) defines as `!N = ...` (numba's
# test_debuginfo.test_DILocation reads them this way). The nodes below are numbered out of print order on
# purpose, so the check cannot pass by coincidence.
module = llvm.parse_assembly("""
define double @half(double %x) !dbg !9 {
  %r = fmul double %x, 5.000000e-01, !dbg !4
  ret double %r, !dbg !4
}
!llvm.module.flags = !{!7}
!llvm.dbg.cu = !{!12}
!4 = !DILocation(line: 42, column: 7, scope: !9)
!7 = !{i32 2, !"Debug Info Version", i32 3}
!9 = distinct !DISubprogram(name: "half", file: !3, line: 41, type: !1, unit: !12, spFlags: DISPFlagDefinition)
!1 = !DISubroutineType(types: !{})
!3 = !DIFile(filename: "half.py", directory: "/")
!12 = distinct !DICompileUnit(language: DW_LANG_C, file: !3, emissionKind: FullDebug)
""")
instruction = next(iter(next(iter(module.get_function("half").blocks)).instructions))
inst_text = str(instruction)  # read the instruction before the module is printed, so the check sees the instruction's own numbering
# The location survived parsing (without the "Debug Info Version" flag the parser drops it and the checks pass vacuously).
assert re.search(r", !dbg ![0-9]+$", inst_text), inst_text
named = re.findall(r"![0-9]+", inst_text)
defined = dict(re.findall(r"^(![0-9]+) = (.*)$", str(module), re.M))
assert set(named) <= set(defined), f"metadata numbering: {inst_text!r} names {sorted(set(named) - set(defined))}, which str(module) never defines"
assert defined[named[-1]].startswith("!DILocation(line: 42, column: 7,"), f"metadata numbering: {inst_text!r} names {named[-1]}, which str(module) defines as {defined[named[-1]]}"
print("OK -- str(instruction) names", named, "and str(module) defines it as", defined[named[-1]])